In [4]:
import numpy as np


def Newton_step(lamb0, dlamb, s0, ds):
    """Step-size correction exactly as given in the project PDF."""
    alp = 1
    idx_lamb0 = np.array(np.where(dlamb < 0))
    if idx_lamb0.size > 0:
        alp = min(alp, np.min(-lamb0[idx_lamb0] / dlamb[idx_lamb0]))

    idx_s0 = np.array(np.where(ds < 0))
    if idx_s0.size > 0:
        alp = min(alp, np.min(-s0[idx_s0] / ds[idx_s0]))

    return alp

In [5]:
def Newton_step_fast(lamb0, dlamb, s0, ds):
    """
    Same result as Newton_step, with less overhead: no np.where index arrays,
    no fancy indexing / copies, no branching on emptiness. Ratios are computed
    in one pass and entries with a non-negative direction are set to +inf, so
    they never win the min.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        a_lamb = np.where(dlamb < 0, -lamb0 / dlamb, np.inf).min()
        a_s = np.where(ds < 0, -s0 / ds, np.inf).min()
    return min(1.0, a_lamb, a_s)

In [6]:
def Newton_step_recip(lamb0, dlamb, s0, ds):
    """
    Same alpha, reformulated using that the iterates are strictly interior
    (lamb0 > 0, s0 > 0), which the 0.95 damping in the update guarantees.

    For v > 0 and dv_i < 0:   -v_i / dv_i = 1 / t_i,   with  t_i = -dv_i / v_i > 0,
    and the minimum of positive reciprocals is the reciprocal of the maximum:

        alpha = min(1, min_{dv_i < 0} -v_i/dv_i) = 1 / max(1, max_i t_i).

    Components with dv_i >= 0 give t_i <= 0 < 1, so they can never beat the 1
    and need no filtering. The result: no sign test, no subset selection, and
    the divisor v_i is always positive (no division by zero, no inf/nan).
    One division per component and one max, instead of compare + select + divide + min.
    """
    t = max(1.0, np.max(-dlamb / lamb0), np.max(-ds / s0))
    return 1.0 / t

In [7]:
def build_KKT(G, A, C, s, lamb):
    """KKT matrix M_KKT for z = (x, gamma, lambda, s). A may have 0 columns."""
    n, p = A.shape
    m = C.shape[1]
    Z = np.zeros
    return np.block([
        [G,          -A,          -C,          Z((n, m))],
        [-A.T,       Z((p, p)),   Z((p, m)),   Z((p, m))],
        [-C.T,       Z((m, p)),   Z((m, m)),   np.eye(m)],
        [Z((m, n)),  Z((m, p)),   np.diag(s),  np.diag(lamb)],
    ])


def residuals(G, g, A, C, b, d, x, gamma, lamb, s):
    rL = G @ x + g - A @ gamma - C @ lamb
    rA = b - A.T @ x
    rC = s + d - C.T @ x
    rs = s * lamb
    return rL, rA, rC, rs


def interior_point(G, g, A, C, b, d, x0, gamma0, lamb0, s0,
                   solve=np.linalg.solve, step=Newton_step_recip,
                   eps=1e-16, maxiter=100):
    """
    Predictor-corrector interior point method (section 1.2).
    `solve(M, rhs)` solves the KKT system (swap it for other strategies).
    `step` is the step-size correction (Newton_step, Newton_step_fast or Newton_step_recip).
    Returns x, gamma, lambda, s and the number of iterations.
    """
    n, p = A.shape
    m = C.shape[1]
    x, gamma, lamb, s = x0.copy(), gamma0.copy(), lamb0.copy(), s0.copy()
    e = np.ones(m)

    def split(v):
        return v[:n], v[n:n + p], v[n + p:n + p + m], v[n + p + m:]

    k = 0
    for k in range(1, maxiter + 1):
        rL, rA, rC, rs = residuals(G, g, A, C, b, d, x, gamma, lamb, s)
        mu = s @ lamb / m

        # stopping criterion
        if (max(np.linalg.norm(rL), np.linalg.norm(rA), np.linalg.norm(rC)) < eps
                or abs(mu) < eps):
            k -= 1
            break

        M = build_KKT(G, A, C, s, lamb)

        # 1. predictor: standard Newton step
        dz = solve(M, -np.concatenate([rL, rA, rC, rs]))
        dx, dg, dl, ds = split(dz)

        # 2. step-size correction
        alpha = step(lamb, dl, s, ds)

        # 3. centering parameter
        mu_bar = (s + alpha * ds) @ (lamb + alpha * dl) / m
        sigma = (mu_bar / mu) ** 3

        # 4. corrector: same matrix, modified last block of the rhs
        rs_hat = rs + ds * dl - sigma * mu * e
        dz = solve(M, -np.concatenate([rL, rA, rC, rs_hat]))
        dx, dg, dl, ds = split(dz)

        # 5. step-size correction
        alpha = step(lamb, dl, s, ds)

        # 6. update
        x = x + 0.95 * alpha * dx
        gamma = gamma + 0.95 * alpha * dg
        lamb = lamb + 0.95 * alpha * dl
        s = s + 0.95 * alpha * ds

    return x, gamma, lamb, s, k


# Quick check on the test problem of section 2.1 (A = 0 -> no equality constraints)
n = 10
m = 2 * n
g = np.random.randn(n)
G = np.eye(n)
C = np.hstack([np.eye(n), -np.eye(n)])
d = -10 * np.ones(m)
A = np.zeros((n, 0))
b = np.zeros(0)

for name, f in [("Newton_step", Newton_step), ("Newton_step_fast", Newton_step_fast),
                ("Newton_step_recip", Newton_step_recip)]:
    x, gamma, lamb, s, k = interior_point(G, g, A, C, b, d,
                                          np.zeros(n), np.zeros(0), np.ones(m), np.ones(m),
                                          step=f)
    print(f"{name}: iterations {k}, max|x+g| {np.max(np.abs(x + g)):.2e}")


Newton_step: iterations 17, max|x+g| 0.00e+00
Newton_step_fast: iterations 17, max|x+g| 0.00e+00
Newton_step_recip: iterations 17, max|x+g| 0.00e+00
